# Vision Transformer

Reimplementation of the full Vision Transformer (ViT) from [An Image is Worth 16x16 Words: Transformers for Image Recognition at Scale](https://arxiv.org/abs/2010.11929) (Dosovitskiy et al., 2020), using only basic PyTorch layers.

<p align="center">
  <img src="../assets/vit.gif" alt="Vision Transformer" width="50%">
</p>

In this notebook I assemble the patch embedding and the Transformer encoder block from the previous notebooks, imported from `src/vit/model.py`, into the complete model:

$$\mathbf{z}_0 = [\mathbf{x}_\text{class};\, \mathbf{x}^1_p \mathbf{E};\, \mathbf{x}^2_p \mathbf{E};\, \cdots;\, \mathbf{x}^N_p \mathbf{E}] + \mathbf{E}_{pos} \qquad \text{(Eq. 1)}$$

$$\mathbf{z}'_\ell = \text{MSA}(\text{LN}(\mathbf{z}_{\ell-1})) + \mathbf{z}_{\ell-1}, \qquad \ell = 1 \dots L \qquad \text{(Eq. 2)}$$

$$\mathbf{z}_\ell = \text{MLP}(\text{LN}(\mathbf{z}'_\ell)) + \mathbf{z}'_\ell, \qquad \ell = 1 \dots L \qquad \text{(Eq. 3)}$$

$$\mathbf{y} = \text{LN}(\mathbf{z}^0_L) \qquad \text{(Eq. 4)}$$

**Contents**
1. Setup
2. ViT model
3. Model summary
4. Forward pass

## 1. Setup

In [1]:
import torch 
import torch.nn as nn

from torchinfo import summary

from vit.model import PatchEmbedding, TransformerEncoderBlock

In [2]:
def set_seed(seed: int = 42):
    """Seeds PyTorch's CPU and CUDA random number generators for reproducible runs.

    Args:
        seed: Seed value passed to the generators.
    """
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)

## 2. ViT model

The class token $\mathbf{x}_\text{class}$ is prepended to the embedded patches and the position embeddings $\mathbf{E}_{pos}$ are added (Eq. 1), followed by dropout (Appendix B.1). The $L$ encoder blocks are stacked with `nn.Sequential`, since each block only consumes the output of the previous one (Eq. 2 and 3).

Only the class token output of the last block, $\mathbf{z}^0_L$, is used for classification. It is normalized (Eq. 4) and mapped to class logits by a single linear layer, the head the paper uses at fine-tuning time (Section 3.1).

### 2.1 `ViT` module

The same class lives in `src/vit/model.py`.

In [3]:
class ViT(nn.Module):
    """Vision Transformer (ViT) model. The defaults build ViT-Base/16 (Table 1) for 256x256 RGB images.

    Args:
        image_size: Side length H=W of the input images.
        patch_size: Side length P of each square patch.
        color_channels: Number of image channels C.
        embedding_dim: Dimension D of the token embeddings.
        num_transformer_layers: Number of Transformer encoder blocks L.
        num_heads: Number of attention heads h.
        mlp_dim: Dimension of the hidden layer in the MLP.
        num_classes: Number of output classes for classification.
        mlp_dropout: Dropout probability applied after the attention output projection and each linear layer of the MLP.
        embedding_dropout: Dropout probability applied after adding the position embeddings to the patch embeddings.
    """

    def __init__(
        self,
        image_size: int = 256,
        patch_size: int = 16,
        color_channels: int = 3,
        embedding_dim: int = 768,
        num_transformer_layers: int = 12,
        num_heads: int = 12,
        mlp_dim: int = 3072,
        num_classes: int = 21,
        mlp_dropout: float = 0.1,
        embedding_dropout: float = 0.1,
    ):
        super().__init__()
        assert image_size % patch_size == 0, f"Image size must be divisible by patch size, image size: {image_size}, patch size: {patch_size}."

        self.num_patches = (image_size * image_size) // patch_size**2

        self.class_embedding = nn.Parameter(torch.zeros((1, 1, embedding_dim), requires_grad=True))
        self.position_embedding = nn.Parameter(torch.randn((1, self.num_patches + 1, embedding_dim), requires_grad=True) * 0.02)
        self.embedding_dropout = nn.Dropout(p=embedding_dropout)
        self.patch_embedding = PatchEmbedding(patch_size, color_channels, embedding_dim)

        self.transformer_encoder = nn.Sequential(*[TransformerEncoderBlock(embedding_dim, num_heads, mlp_dim, mlp_dropout) for _ in range(num_transformer_layers)])

        self.classifier = nn.Sequential(
            nn.LayerNorm(normalized_shape=embedding_dim),
            nn.Linear(in_features=embedding_dim, out_features=num_classes)
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Computes the output logits of the ViT model.

        Args:
            x: Tensor of shape (B, C, H, W).

        Returns:
            Tensor of shape (B, num_classes).
        """
        batch_size = x.shape[0]
        patch_embeddings = self.patch_embedding(x)
        class_embeddings = self.class_embedding.expand(batch_size, -1, -1)
        embeddings = torch.cat((class_embeddings, patch_embeddings), dim=1)
        embeddings += self.position_embedding
        embeddings = self.embedding_dropout(embeddings)

        transformer_output = self.transformer_encoder(embeddings)
        class_token_output = transformer_output[:, 0]
        logits = self.classifier(class_token_output)

        return logits

## 3. Model summary

With the ViT-Base sizes from Table 1 ($L = 12$, $D = 768$, MLP size $3072$, $h = 12$) and $P = 16$, a $256 \times 256$ image becomes $N = 256$ patches plus the class token.

In [4]:
set_seed()

vit = ViT(num_classes=21)

summary(
    model=vit,
    input_size=(32, 3, 256, 256), # (batch_size, color_channels, height, width)
    col_names=["input_size", "output_size", "num_params", "trainable"],
    col_width=20,
    row_settings=["var_names"]
)

Layer (type (var_name))                                           Input Shape          Output Shape         Param #              Trainable
ViT (ViT)                                                         [32, 3, 256, 256]    [32, 21]             198,144              True
├─PatchEmbedding (patch_embedding)                                [32, 3, 256, 256]    [32, 256, 768]       --                   True
│    └─Linear (linear)                                            [32, 256, 768]       [32, 256, 768]       590,592              True
├─Dropout (embedding_dropout)                                     [32, 257, 768]       [32, 257, 768]       --                   --
├─Sequential (transformer_encoder)                                [32, 257, 768]       [32, 257, 768]       --                   True
│    └─TransformerEncoderBlock (0)                                [32, 257, 768]       [32, 257, 768]       --                   True
│    │    └─MultiHeadAttentionBlock (mha_block)            

## 4. Forward pass

A random image checks that the forward pass returns one logit per class. The model is untrained, so the predicted class is meaningless.

In [5]:
random_image_tensor = torch.randn(1, 3, 256, 256)

preds = vit(random_image_tensor)

print(f"Shape of the output tensor: {preds.shape}")
print(f"Output tensor: {preds}")
print(f"Output tensor (argmax): {preds.argmax(dim=1)}")

Shape of the output tensor: torch.Size([1, 21])
Output tensor: tensor([[-0.5201, -0.4961, -0.4850,  0.1728,  0.3864, -0.8500, -0.2165, -0.5106,
         -0.9345, -0.1955,  0.7438, -0.0646,  0.1455, -0.5587, -0.2728, -0.7624,
          0.1455,  0.7776,  0.9182, -0.2438,  0.0017]],
       grad_fn=<AddmmBackward0>)
Output tensor (argmax): tensor([18])
